# 04 — Live Match Tracker
Enter actual match results and compare against model predictions.

Metrics: accuracy %, MAE scoreline error, per-match prediction badges.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

DATA_DIR = Path('../data')
RESULTS_FILE = DATA_DIR / 'actual_results.csv'

bundle    = joblib.load(DATA_DIR / 'model.pkl')
clf       = bundle['clf']
home_pipe = bundle['home_pipe']
away_pipe = bundle['away_pipe']
le        = bundle['le']
XFEATS    = bundle['xfeats']
feat_idx  = bundle['feat_idx']

# 2026 World Cup teams
ALL_TEAMS_2026 = sorted([
    'Mexico', 'South Africa', 'South Korea', 'Czech Republic',
    'Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland',
    'Brazil', 'Morocco', 'Haiti', 'Scotland',
    'United States', 'Paraguay', 'Australia', 'Turkey',
    'Germany', 'Cote dIvoire', 'Thailand', 'Chile',
    'Spain', 'Egypt', 'Tunisia', 'Costa Rica',
    'England', 'Croatia', 'Ghana', 'Panama',
    'France', 'Senegal', 'Iraq', 'Norway',
    'Argentina', 'Algeria', 'Austria', 'Jordan',
    'Portugal', 'DR Congo', 'Uzbekistan', 'Colombia',
    'Netherlands', 'Iran', 'New Zealand', 'Japan',
    'Belgium', 'Uruguay', 'Sweden', 'Saudi Arabia',
])

print('Model loaded. Validation accuracy:', f"{bundle['val_accuracy']:.1%}")

Model loaded. Validation accuracy: 50.0%


## Enter actual results
Edit the list below and re-run the cell to add more results.

In [2]:
# Format: (home_team, away_team, home_goals, away_goals, match_date, stage)
ACTUAL_RESULTS = [
    # --- Example entries (replace with real 2026 results) ---
    # ('United States', 'Panama',   2, 0, '2026-06-12', 'Group A'),
    # ('Argentina',     'Australia', 2, 1, '2026-06-13', 'Group B'),
]

if ACTUAL_RESULTS:
    actual_df = pd.DataFrame(ACTUAL_RESULTS,
        columns=['home_team','away_team','home_goals','away_goals','match_date','stage'])
else:
    # Load from file if it exists
    if RESULTS_FILE.exists():
        actual_df = pd.read_csv(RESULTS_FILE)
        print(f'Loaded {len(actual_df)} results from file.')
    else:
        actual_df = pd.DataFrame(columns=['home_team','away_team','home_goals','away_goals','match_date','stage'])
        print('No results yet. Add them above and re-run.')

print(f'Tracking {len(actual_df)} matches.')

No results yet. Add them above and re-run.
Tracking 0 matches.


## Generate predictions for entered matches

In [3]:
FEATURE_COLS = ['win_rate','avg_goals_for','avg_goals_against','goal_diff',
                'avg_xgf','avg_xga','rank_score']
medians = feat_idx[FEATURE_COLS].median() if len(feat_idx) > 0 else pd.Series({c: 0 for c in FEATURE_COLS})

def get_team_feat(team, prefix):
    if team in feat_idx.index:
        row = feat_idx.loc[team]
    else:
        row = medians
    return {f'{prefix}_{c}': float(row.get(c, 0)) for c in FEATURE_COLS}

def predict_match(home, away):
    h = get_team_feat(home, 'h')
    a = get_team_feat(away, 'a')
    rk_h = feat_idx['rank_score'].get(home, 0) if 'rank_score' in feat_idx.columns else 0
    rk_a = feat_idx['rank_score'].get(away, 0) if 'rank_score' in feat_idx.columns else 0
    row = pd.DataFrame([{**h, **a, 'rank_diff': rk_h - rk_a}])[XFEATS].fillna(0)
    probs  = clf.predict_proba(row)[0]
    outcome = dict(zip(le.classes_, probs))
    mu_h   = max(float(home_pipe.predict(row)[0]), 0.1)
    mu_a   = max(float(away_pipe.predict(row)[0]), 0.1)
    pred_result = max(outcome, key=outcome.get)
    return {
        'pred_result': pred_result,
        'prob_home': outcome.get('H', 0),
        'prob_draw': outcome.get('D', 0),
        'prob_away': outcome.get('A', 0),
        'pred_home_goals': round(mu_h, 1),
        'pred_away_goals': round(mu_a, 1),
    }

if len(actual_df) > 0:
    pred_rows = actual_df.apply(lambda r: predict_match(r['home_team'], r['away_team']), axis=1)
    pred_df = pd.DataFrame(list(pred_rows))
    tracker = pd.concat([actual_df.reset_index(drop=True), pred_df], axis=1)

    def actual_result(row):
        if row['home_goals'] > row['away_goals']:   return 'H'
        elif row['home_goals'] < row['away_goals']: return 'A'
        else:                                       return 'D'

    tracker['actual_result'] = tracker.apply(actual_result, axis=1)
    tracker['correct'] = tracker['pred_result'] == tracker['actual_result']

    # Score MAE
    tracker['home_score_err'] = abs(tracker['home_goals'] - tracker['pred_home_goals'])
    tracker['away_score_err'] = abs(tracker['away_goals'] - tracker['pred_away_goals'])
    tracker['score_mae'] = (tracker['home_score_err'] + tracker['away_score_err']) / 2

    # Badge
    def badge(row):
        if row['correct']:
            if row['score_mae'] < 0.5: return '🟢 Exact'
            return '🟡 Result'
        return '🔴 Wrong'
    tracker['badge'] = tracker.apply(badge, axis=1)

    tracker.to_csv(RESULTS_FILE, index=False)
    print(f'Results saved to {RESULTS_FILE}')
    display(tracker[['home_team','away_team','home_goals','away_goals',
                      'pred_home_goals','pred_away_goals','pred_result',
                      'actual_result','badge']].to_string(index=False))
else:
    print('Add results above to begin tracking.')

Add results above to begin tracking.


## Accuracy & MAE metrics

In [4]:
if len(actual_df) > 0 and 'correct' in tracker.columns:
    accuracy = tracker['correct'].mean()
    mae      = tracker['score_mae'].mean()
    n        = len(tracker)

    html = f"""
    <div style='font-family:sans-serif; padding:16px; background:#1a1a2e; color:white; border-radius:8px'>
      <h2 style='margin:0 0 12px'>📊 Prediction Scorecard</h2>
      <table style='border-collapse:collapse; width:100%'>
        <tr><td style='padding:8px; color:#aaa'>Matches tracked</td><td style='padding:8px; font-weight:bold'>{n}</td></tr>
        <tr><td style='padding:8px; color:#aaa'>Result accuracy</td><td style='padding:8px; font-weight:bold; color:{"#2ecc71" if accuracy > 0.5 else "#e74c3c"}'>{accuracy:.1%}</td></tr>
        <tr><td style='padding:8px; color:#aaa'>Scoreline MAE</td><td style='padding:8px; font-weight:bold'>{mae:.2f} goals</td></tr>
        <tr><td style='padding:8px; color:#aaa'>Model baseline (2022 WC)</td><td style='padding:8px; color:#888'>{bundle['val_accuracy']:.1%}</td></tr>
      </table>
    </div>
    """
    display(HTML(html))
    
    # Badge breakdown
    print('\nBadge breakdown:')
    print(tracker['badge'].value_counts().to_string())
else:
    print('No results to score yet.')

No results to score yet.
